In [1]:
import re
import pandas as pd
from datasets import load_dataset, concatenate_datasets
import numpy as np
import evaluate
from transformers import (
    BertTokenizer,
    BertGenerationEncoder,
    BertGenerationDecoder,
    EncoderDecoderModel,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [2]:
import sys
print(sys.executable)

/Users/Admin/Desktop/MSE 446/Title-Generator/.venv/bin/python


In [3]:
#UTILS AND PARSERS

class Config:
    CHUNK_CHARS = 1000
    INPUT_PREFIX = "generate title: "
    ENABLE_PROFANITY_FILTER = True
    BLOCKED_WORDS = ["fuck", "shit", "bitch", "cunt", "slut", "ass", "dick"] 

cfg = Config()

# --- Safety & Formatting Utilities ---

try:
    from better_profanity import profanity
    profanity.load_censor_words()
    _HAS_PROFANITY = True
except ImportError:
    _HAS_PROFANITY = False

_MAX_PROFANITY_CHARS = 512
_BLOCK_RE = re.compile(r"\b(" + "|".join(re.escape(w) for w in cfg.BLOCKED_WORDS) + r")\b", re.I)

def is_safe_text(text: str) -> bool:
    if not cfg.ENABLE_PROFANITY_FILTER or not text:
        return True
    if _BLOCK_RE.search(text):
        return False
    if _HAS_PROFANITY and len(text) <= _MAX_PROFANITY_CHARS:
        return not profanity.contains_profanity(text)
    return True

def clean_and_format(text: str) -> str:
    text = text.strip()
    truncated = text if len(text) <= cfg.CHUNK_CHARS else text[:cfg.CHUNK_CHARS]
    return cfg.INPUT_PREFIX + truncated

def is_descriptive_title(title: str) -> bool:
    title_lower = title.lower().strip()
    
    # Clean out digits and common punctuation
    clean_text = re.sub(r'[^a-z\s]', ' ', title_lower)
    words = clean_text.split()
    
    if not words: 
        return False
        
    # Expanded to include plurals and serialization terms
    generic_keywords = {
        "chapter", "chapters", "scene", "scenes", "act", "acts", 
        "part", "parts", "book", "books", "prologue", "epilogue", 
        "volume", "volumes", "section", "sections", "story", "stories",
        "untitled", "i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x"
    }
    
    if all(word in generic_keywords for word in words):
        return False
        
    return True

# --- Dataset Parsers ---

def process_cmu(batch):
    titles, texts, valid = [], [], []
    for title, summary in zip(batch["title"], batch["summary"]):
        title, summary = (title or "").strip(), (summary or "").strip()
        
        is_ok = (
            len(summary) >= 100 
            and 2 <= len(title) <= 120 
            and is_descriptive_title(title) # Applied here
            and is_safe_text(summary[:cfg.CHUNK_CHARS])
            and is_safe_text(title)
        )
        
        titles.append(title)
        texts.append(clean_and_format(summary) if is_ok else "")
        valid.append(is_ok)
        
    return {"title": titles, "text": texts, "is_valid": valid, "source": ["cmu_book_summaries"] * len(valid)}

def process_booksum(batch, use_summary=True):
    titles, texts, valid = [], [], []
    narrative_key = "summary_text" if use_summary else "chapter"
    
    for title, raw_narrative in zip(batch["summary_name"], batch[narrative_key]):
        title, narrative = (title or "").strip(), (raw_narrative or "").strip()
        
        is_ok = (
            len(narrative) >= 100 
            and 2 <= len(title) <= 120
            and is_descriptive_title(title) # Applied here
            and is_safe_text(narrative[:cfg.CHUNK_CHARS])
            and is_safe_text(title)
        )
        
        titles.append(title)
        texts.append(clean_and_format(narrative) if is_ok else "")
        valid.append(is_ok)
        
    return {"title": titles, "text": texts, "is_valid": valid, "source": ["novel_chapter_booksum"] * len(valid)}

def process_gutenberg(batch):
    titles, texts, valid = [], [], []
    
    for raw_text in batch["text"]:
        raw_text = (raw_text or "").strip()
        match = re.match(r"^([A-Z0-9\s.,;:!?'-]+?)\n+(.*)", raw_text, flags=re.DOTALL)
        
        is_ok = False
        title, body = "", ""
        if match:
            # Strip trailing punctuation like semicolons from the captured title
            title = match.group(1).strip().strip(";:.,-")
            body = match.group(2).strip()
            
            body_start = body.lower()[:300]
            
            # Catch title pages AND serial collection markers (e.g., "No. 1", "Vol. 2")
            is_metadata = (
                "author of" in body_start or 
                "by " in body_start or 
                "transcribed by" in body_start or 
                "reprinted" in body_start or
                re.match(r"^(no\.\s*\d+|vol\.\s*\d+|volume\s*\d+)", body_start)
            )
            
            if (title.isupper() 
                and len(title) > 2 
                and not is_metadata
                and is_descriptive_title(title) 
                and is_safe_text(body[:cfg.CHUNK_CHARS]) 
                and is_safe_text(title)):
                is_ok = True
                
        titles.append(title)
        texts.append(clean_and_format(body) if is_ok else "")
        valid.append(is_ok)
        
    return {"title": titles, "text": texts, "is_valid": valid, "source": ["gutenberg_chapters"] * len(valid)}

In [4]:
#LOAD DATASETS
def pipeline_load_and_clean(dataset_id, process_fn, config_name=None, split="train"):
    print(f"\nLoading and processing {dataset_id}...")
    ds = load_dataset(dataset_id, config_name, split=split) if config_name else load_dataset(dataset_id, split=split)
    processed = ds.map(process_fn, batched=True, remove_columns=ds.column_names)
    filtered = processed.filter(lambda x: x["is_valid"] is True).remove_columns(["is_valid"])
    
    print(f" -> Retained {len(filtered)} valid rows from {dataset_id}") 
    return filtered

if __name__ == "__main__":
    cmu_ds = pipeline_load_and_clean("textminr/cmu-book-summaries", process_cmu)
    booksum_ds = pipeline_load_and_clean("kmfoda/booksum", lambda b: process_booksum(b, use_summary=True))
    gutenberg_ds = pipeline_load_and_clean("zkeown/gutenberg-corpus", process_gutenberg, config_name="chapters")
    
    print("\nConcatenating and compounding datasets...")
    final_dataset = concatenate_datasets([cmu_ds, booksum_ds, gutenberg_ds]).shuffle(seed=42)
    
    if len(gutenberg_ds) > 0:
        print("\n--- Gutenberg Extraction ---")
        display_gutenberg = pd.DataFrame(gutenberg_ds.select(range(1)))
        print(display_gutenberg[["source", "title", "text"]])
    
    print("\n--- Final Processed Dataset Preview (Shuffled) ---")
    display_df = pd.DataFrame(final_dataset.select(range(10)))
    print(display_df[["source", "title", "text"]])


Loading and processing textminr/cmu-book-summaries...


Map:   0%|          | 0/16559 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16559 [00:00<?, ? examples/s]

 -> Retained 15665 valid rows from textminr/cmu-book-summaries

Loading and processing kmfoda/booksum...


Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9600 [00:00<?, ? examples/s]

 -> Retained 1235 valid rows from kmfoda/booksum

Loading and processing zkeown/gutenberg-corpus...


Map:   0%|          | 0/305 [00:00<?, ? examples/s]

Filter:   0%|          | 0/305 [00:00<?, ? examples/s]

 -> Retained 115 valid rows from zkeown/gutenberg-corpus

Concatenating and compounding datasets...

--- Gutenberg Extraction ---
               source              title  \
0  gutenberg_chapters  A GOODLY HERITAGE   

                                                text  
0  generate title: "It is not best in an inglorio...  

--- Final Processed Dataset Preview (Shuffled) ---
               source                                              title  \
0  cmu_book_summaries  The Thirteen and a Half Lives of Captain Bluebear   
1  cmu_book_summaries                                  The October Horse   
2  cmu_book_summaries                                             Lilith   
3  cmu_book_summaries                                   The Green Knight   
4  cmu_book_summaries                               The Heidi Chronicles   
5  cmu_book_summaries                           Clear and Present Danger   
6  cmu_book_summaries                                        Hadji Murat   
7  cmu_book

In [5]:
#TOKENIZER
print("\nInitializing tokenizer...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(batch):
    inputs = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)
    targets = tokenizer(batch["title"], padding="max_length", truncation=True, max_length=32)
    return {
        "input_ids": inputs.input_ids,
        "attention_mask": inputs.attention_mask,
        "labels": targets.input_ids
    }

print("Tokenizing dataset (this may take a moment)...")
tokenized_dataset = final_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
# Select a random subset of 10,000 rows for rapid training
tokenized_dataset = tokenized_dataset.shuffle(seed=42).select(range(10000))

# Split for evaluation
split_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)


Initializing tokenizer...
Tokenizing dataset (this may take a moment)...


Map:   0%|          | 0/17015 [00:00<?, ? examples/s]

In [6]:
#MODEL INITIALIZATION
print("\nInitializing BertGeneration Seq2Seq Model...")

encoder = BertGenerationEncoder.from_pretrained(
    "bert-base-uncased", 
    bos_token_id=tokenizer.cls_token_id, 
    eos_token_id=tokenizer.sep_token_id
)

decoder = BertGenerationDecoder.from_pretrained(
    "bert-base-uncased", 
    add_cross_attention=True, 
    is_decoder=True, 
    bos_token_id=tokenizer.cls_token_id, 
    eos_token_id=tokenizer.sep_token_id
)

model = EncoderDecoderModel(encoder=encoder, decoder=decoder)

# Configure generation parameters
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.encoder.vocab_size
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.bos_token_id = tokenizer.cls_token_id

[transformers] You are using a model of type `bert` to instantiate a model of type `bert-generation`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.



Initializing BertGeneration Seq2Seq Model...


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] BertGenerationEncoder LOAD REPORT from: bert-base-uncased
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
bert.embeddings.token_type_embeddings.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                    | UNEXPECTED |  | 
cls.predictions.bias                         | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias         | UNEXPECTED |  | 
bert.pooler.dense.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight       | UNEXPECTED |  | 
bert.pooler.dense.weight                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias     | UNEXPECTED |  | 
cls.seq_relationship.weight                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model o

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] This checkpoint seem corrupted. The tied weights mapping for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are absent from the checkpoint, and we could not find another related tied weight for those keys
[transformers] BertGenerationDecoder LOAD REPORT from: bert-base-uncased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.embeddings.token_type_embeddings.weight                       | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
cls.predictions.bias                                               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight                         | UNEXPECTED | 
cls.predictions.transform.dense.bias                               | UNEXPECTED | 
bert.pooler.dense.bias                                             | UNEXPECTED | 
cls.predictions

In [7]:
#EVALUATION METRICS
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    if isinstance(predictions, tuple):
        predictions = predictions[0]
        
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = ["\n".join(pred.strip().split()) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split()) for label in decoded_labels]
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

In [ ]:
#TRAINING LOOP
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

# Split for evaluation
split_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bert-title-generator",
    eval_strategy="steps",
    eval_steps=500,
    logging_strategy="steps",
    logging_steps=100,
    save_strategy="steps",
    save_steps=500,
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,       
    dataloader_pin_memory=False,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=False,                   
    report_to="none"
)

print("\nInitializing Seq2SeqTrainer...")
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting training loop...")
trainer.train()

print("Training complete. Saving final model...")
trainer.save_model("./bert-title-generator-final")
tokenizer.save_pretrained("./bert-title-generator-final")

In [ ]:
#TEST FUNCTION
def generate_chapter_title(chapter_text: str, model_dir: str = "./bert-title-generator-final") -> str:
    """
    Loads the trained BertGeneration model and generates a title for the provided text.
    """
    # 1. Load the tokenizer and model from your saved directory
    tokenizer = BertTokenizer.from_pretrained(model_dir)
    model = EncoderDecoderModel.from_pretrained(model_dir)

    # 2. Assign to Mac's MPS (GPU) if available, otherwise CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    
    # Put the model in evaluation mode (disables dropout layers, etc.)
    model.eval() 

    # 3. Format the text with the exact prefix used during training
    formatted_text = "generate title: " + chapter_text.strip()

    # 4. Tokenize the input (truncating to BERT's 512 limit)
    inputs = tokenizer(
        formatted_text,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt" # Return PyTorch tensors
    ).to(device)

    # 5. Generate the output tokens
    # Using torch.no_grad() saves memory since we aren't calculating gradients for training
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=32,            # The max length we set for titles
            num_beams=4,              # Uses beam search to find the highest probability sequence
            early_stopping=True,      # Stops once the model predicts an EOS token
            no_repeat_ngram_size=2    # Prevents the model from repeating the same phrases
        )

    # 6. Decode the generated tensor back into a human-readable string
    generated_title = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Optional: Capitalize the title neatly
    return generated_title.title()

In [ ]:
test_text_1 = """
The airlock hissed open, revealing the desolate, crimson landscape of Kepler-186f. 
Captain Vance adjusted his rebreather, the mechanical hum the only sound breaking 
the absolute silence of the dead colony. He stepped out onto the rusted grating, 
his flashlight cutting through the thick, settling dust. They were supposed to 
find three hundred sleeping colonists here. Instead, they found empty cryo-pods 
and blast doors sealed from the outside.
"""

print("Test 1 Result:", generate_chapter_title(test_text_1))

test_text_2 = """
Elara pulled the heavy velvet cloak tight around her shoulders as the carriage 
rattled over the cobblestones. The spires of the Iron Keep loomed in the distance, 
silhouetted against a bruising purple sky. In her pocket, the glass vial felt heavy, 
the glowing liquid inside pulsing in time with her racing heartbeat. If the King 
drank it, the curse would break. If she was caught, the executioner's block awaited.
"""

print("Test 2 Result:", generate_chapter_title(test_text_2))